In [ ]:
from pathlib import Path
from PIL import Image, ImageOps

RAW = Path("originals")
PROC = Path("processed")
SIZE = 224

skipped = []
for p in RAW.rglob("*"):
    if not p.is_file():
        continue
    if p.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
        skipped.append(p.name)
        continue

    img = Image.open(p)
    img = ImageOps.exif_transpose(img)
    img = img.convert("RGB")

    w, h = img.size
    s = min(w, h)
    left = (w - s) // 2
    top = (h - s) // 2
    img = img.crop((left, top, left + s, top + s))
    img = img.resize((SIZE, SIZE), Image.LANCZOS)

    out = (PROC / p.relative_to(RAW)).with_suffix(".jpg")
    out.parent.mkdir(parents=True, exist_ok=True)
    img.save(out, quality=95)

print("Skipped files:", len(skipped), set(Path(s).suffix for s in skipped))